# GenDiff tutorial 1 — preprocessing and training

GenDiff learns a **displacement field** ΔX over single-cell state space: for every cell it predicts the
expression change that carries the cell forward along its differentiation trajectory under a given
perturbation. This notebook goes from a raw-ish `AnnData` to a trained, saved model using the high-level
`gendiff.GenDiff` API:

1. get the four inputs GenDiff needs (log-expression, a low-dim representation, a pseudotime, a condition column),
2. `GenDiff.setup_anndata(...)` — build the ΔX supervision target with the knn sampler,
3. `GenDiff(adata)` + `model.train(...)`,
4. `model.save(...)`.

Use the **BarRNA-seq mESC** dataset here (small, dense conditions). Swap in your own h5ad by editing the
load cell. **Note:** this is a runnable template — it has not been executed in the repo. Training needs a
GPU; a ~2k-cell dataset trains in a few minutes on one GPU.

## 0. Environment

In [1]:
import sys, os
REPO = "/rds/user/wz369/hpc-work/GenDiff"
if REPO not in sys.path:
    sys.path.insert(0, REPO)          # so `import gendiff`, `gendiff_dev`, `src` resolve
os.chdir(REPO)

import numpy as np
import pandas as pd
import scanpy as sc
from gendiff import GenDiff

sc.settings.verbosity = 1
np.random.seed(0)

## 1. Load data

The dataset registry resolves a name to an h5ad path. `barrnaseq` is the integrated mESC dataset
(2000 genes, condition column `condition` with a `ctrl` baseline). To use your own data instead,
replace this cell with `adata = sc.read_h5ad("your_file.h5ad")`.

In [2]:
from gendiff_dev.data import registry as data_reg

adata = data_reg.load("barrnaseq")     # data/integrated_mesc_group0_Nov7.h5ad
adata

AnnData object with n_obs × n_vars = 5745 × 2000
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'RA', 'Wnt', 'TgfB', 'Bmp', 'Fgf', 'Notch', 'Shh', 'assignment', 'starting.state', 'dataset', 'control', 'cell_type', 'condition', 'dose_val', 'batch', 'dpt_groups', 'dpt_order', 'dpt_order_indices', 'dpt_pseudotime', 'RA+Wnt+Fgf', 'numeric_iloc', 'split', 'discrete_time'
    var: 'gene_name', 'Gene_Symbol'
    uns: 'condition_colors', 'diff', 'diffmap_evals', 'dpt_changepoints', 'dpt_groups_colors', 'dpt_grouptips', 'iroot', 'neighbors', 'umap', 'unique_token_dict'
    obsm: 'Tr_SampledX_r100', 'X_diffmap', 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'diff_connectivities', 'diff_distances', 'distances'

## 2. What GenDiff needs

`setup_anndata` requires four things. Inspect what is already present; the next cells compute anything
that is missing.

| Requirement | Where | How to make it |
|---|---|---|
| Normalized log-expression | `adata.X` | `normalize_total` + `log1p` |
| Low-dim representation | `adata.obsm[use_rep]` | `sc.pp.pca` (or a batch-corrected rep) |
| Pseudotime | `adata.obs[pseudotime_key]` | `sc.tl.dpt` with a control cell as root |
| Condition column | `adata.obs[condition_key]` | already in your metadata |

In [3]:
print("X dtype:", adata.X.dtype, "| min/max:", float(adata.X.min()), float(adata.X.max()))
print("obsm keys:", list(adata.obsm))
print("relevant obs:", [c for c in ["condition", "dpt_pseudotime", "split", "batch"] if c in adata.obs])
print("\ncondition counts:\n", adata.obs["condition"].value_counts().head())

X dtype: float64 | min/max: -8.75007274428746 10.0
obsm keys: ['Tr_SampledX_r100', 'X_diffmap', 'X_pca', 'X_umap']
relevant obs: ['condition', 'dpt_pseudotime', 'split', 'batch']

condition counts:
 condition
ctrl           384
RA+TgfB+Fgf    328
RA+Fgf         296
RA+TgfB        276
RA+ctrl        234
Name: count, dtype: int64


### 2a. Normalize + log (only if `X` is raw counts)

GenDiff's input and its ΔX target both live in `adata.X`, so it should be normalized log-expression.
BarRNA-seq is already normalized, so the guard below makes this a no-op; keep it for your own raw data.

In [4]:
if float(adata.X.max()) > 50:          # crude "looks like counts" check
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    print("normalized + log1p")
else:
    print("X already looks normalized; skipping")

X already looks normalized; skipping


### 2b. Low-dim representation for the kNN graph

The sampler builds a nearest-neighbour graph in `adata.obsm[use_rep]`. PCA is fine; if you have a batch
covariate, a batch-corrected representation (e.g. Harmony) gives a cleaner manifold.

In [5]:
if "X_pca" not in adata.obsm:
    sc.pp.pca(adata, n_comps=50)

use_rep = "X_pca"
# If you have a batch column and scanpy-external/harmonypy installed, prefer a corrected rep:
#   import scanpy.external as sce
#   sce.pp.harmony_integrate(adata, "batch")
#   use_rep = "X_pca_harmony"
print("using representation:", use_rep, adata.obsm[use_rep].shape)

using representation: X_pca (5745, 30)


### 2c. Pseudotime with a control cell as root

The pseudotime is the differentiation axis. The sampler defines ΔX as the move toward *higher*
pseudotime, so controls must sit **low**. Rooting diffusion pseudotime at a control cell enforces that.

In [6]:
if "dpt_pseudotime" not in adata.obs:
    sc.pp.neighbors(adata, use_rep=use_rep)
    sc.tl.diffmap(adata)
    ctrl = (adata.obs["condition"].astype(str) == "ctrl").values
    adata.uns["iroot"] = int(np.where(ctrl)[0][0])   # any control cell
    sc.tl.dpt(adata)
    print("computed dpt_pseudotime rooted at a control cell")
else:
    print("dpt_pseudotime already present")

dpt_pseudotime already present


## 3. `setup_anndata` — build the ΔX target

This registers the fields and runs the knn sampler **once** to write the supervision target into
`adata.obsm["gendiff_dX"]`. Key arguments:

- `sampler_config="config1"` — geodesic, M=15, repeat=3 (the recommended r3 setting). Other options:
  `"config2"` (euclid, M=30, repeat=3) or `"others"` with `sampler_kwargs={"metric":..,"M":..,"repeat":..}`.
- `same_condition="auto"` — whether neighbours must share the perturbation. `"auto"` turns this on for
  densely-sampled conditions (BarRNA-seq), off for sparse ones.
- `control="ctrl"` — the baseline label, mapped to token 0 (inferred if you omit it).

The sampler prints the exact settings it runs and how many cells had no eligible higher-pseudotime
neighbour (those get ΔX = 0).

In [7]:
GenDiff.setup_anndata(
    adata,
    condition_key="condition",
    pseudotime_key="dpt_pseudotime",
    use_rep=use_rep,
    control="ctrl",
    sampler_config="config1",
    same_condition="auto",
)

[GenDiff.setup] same_condition='auto' -> True (median 158 cells/condition over 32 conditions)


[knn_sampler] preset config='config1' -> metric=geodesic M=15 repeat=3 | same_condition=True use_rep='X_pca' graph_k=15 alpha=1.0


[knn_sampler] built ΔX 5745x2000; 32/5745 requested cells had no eligible higher-pseudotime neighbour -> ΔX=0


[GenDiff.setup] registered: 2000 genes, 32 condition tokens (control='ctrl'), target=obsm['gendiff_dX'], split='gendiff_split'


AnnData object with n_obs × n_vars = 5745 × 2000
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'RA', 'Wnt', 'TgfB', 'Bmp', 'Fgf', 'Notch', 'Shh', 'assignment', 'starting.state', 'dataset', 'control', 'cell_type', 'condition', 'dose_val', 'batch', 'dpt_groups', 'dpt_order', 'dpt_order_indices', 'dpt_pseudotime', 'RA+Wnt+Fgf', 'numeric_iloc', 'split', 'discrete_time', 'gendiff_split'
    var: 'gene_name', 'Gene_Symbol'
    uns: 'condition_colors', 'diff', 'diffmap_evals', 'dpt_changepoints', 'dpt_groups_colors', 'dpt_grouptips', 'iroot', 'neighbors', 'umap', 'unique_token_dict', 'gendiff_token_dict', 'gendiff_setup'
    obsm: 'Tr_SampledX_r100', 'X_diffmap', 'X_pca', 'X_umap', 'gendiff_dX'
    varm: 'PCs'
    obsp: 'connectivities', 'diff_connectivities', 'diff_distances', 'distances'

### Inspect the built target

`setup_anndata` writes `obsm["gendiff_dX"]` (the ΔX target), `obs["discrete_time"]` (pseudotime binned
into the diffusion time index), a `gendiff_split` train/val/test column, and the registry under
`uns["gendiff_setup"]` / `uns["gendiff_token_dict"]`.

In [8]:
dX = np.asarray(adata.obsm["gendiff_dX"])
print("ΔX shape:", dX.shape, "| mean|abs|:", round(float(np.abs(dX).mean()), 4))
frac_zero = float((np.abs(dX).sum(1) == 0).mean())
print("fraction of cells with ΔX = 0:", round(frac_zero, 3))
print("\nsetup registry:")
for k, v in adata.uns["gendiff_setup"].items():
    print(f"  {k}: {v}")
print("\ncondition tokens:", adata.uns["gendiff_token_dict"])

ΔX shape: (5745, 2000) | mean|abs|: 0.5336
fraction of cells with ΔX = 0: 0.006

setup registry:
  condition_key: condition
  pseudotime_key: dpt_pseudotime
  use_rep: X_pca
  layer: X
  split_key: gendiff_split
  batch_key: None
  control: ctrl
  delimiter: |
  sampler_config: config1
  same_condition: True
  target_obsm: gendiff_dX
  gene_dim: 2000
  n_base_perturbs: 32
  max_multiplexing: 1
  time_key: discrete_time

condition tokens: {'ctrl': 0, 'Bmp+Fgf': 1, 'Bmp+ctrl': 2, 'Fgf+ctrl': 3, 'RA+Bmp': 4, 'RA+Bmp+Fgf': 5, 'RA+Fgf': 6, 'RA+TgfB': 7, 'RA+TgfB+Bmp': 8, 'RA+TgfB+Bmp+Fgf': 9, 'RA+TgfB+Fgf': 10, 'RA+Wnt': 11, 'RA+Wnt+Bmp': 12, 'RA+Wnt+Bmp+Fgf': 13, 'RA+Wnt+Fgf': 14, 'RA+Wnt+TgfB': 15, 'RA+Wnt+TgfB+Bmp': 16, 'RA+Wnt+TgfB+Bmp+Fgf': 17, 'RA+Wnt+TgfB+Fgf': 18, 'RA+ctrl': 19, 'TgfB+Bmp': 20, 'TgfB+Bmp+Fgf': 21, 'TgfB+Fgf': 22, 'TgfB+ctrl': 23, 'Wnt+Bmp': 24, 'Wnt+Bmp+Fgf': 25, 'Wnt+Fgf': 26, 'Wnt+TgfB': 27, 'Wnt+TgfB+Bmp': 28, 'Wnt+TgfB+Bmp+Fgf': 29, 'Wnt+TgfB+Fgf': 30, 'Wnt+ctrl

**Sanity check:** how does the displacement magnitude vary along the trajectory? Binning |ΔX| by
pseudotime summarises the field. The exact shape is dataset-dependent — it need not be monotone (on this
dataset it stays roughly flat).

In [9]:
pt = adata.obs["dpt_pseudotime"].astype(float).values
mag = np.linalg.norm(dX, axis=1)
bins = pd.qcut(pt, 5, labels=False, duplicates="drop")
print(pd.DataFrame({"|dX|": mag}).groupby(bins).mean().rename_axis("pseudotime quintile"))

                          |dX|
pseudotime quintile           
0                    41.325165
1                    43.864880
2                    43.555389
3                    44.044357
4                    45.968269


## 4. Build the model

`GenDiff(adata, ...)` constructs the epsilon net + diffusion learner from the registered setup.
`hidden_size` is the MLP width schedule; `condition_emb_dim` (default 256) becomes the latent layer and
should appear in `hidden_size` (otherwise it is inserted with a warning).

In [10]:
model = GenDiff(
    adata,
    hidden_size=(512, 512, 256, 512, 512),
    condition_emb_dim=256,
    time_emb_dim=64,
    timesteps=200,
)

/rds/user/wz369/hpc-work/LIBS/mamba/envs/GenDiff_env/lib/python3.9/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/rds/user/wz369/hpc-work/LIBS/mamba/envs/GenDiff_env/lib/python3.9/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assi

[GenDiff] Epsilon_Linear + Equivalent_Diffuse_Sampler: 2.89M params, timesteps=200


## 5. Train

Wraps a PyTorch-Lightning Trainer, monitors `val_loss`, and checkpoints the best epoch.
`accelerator="auto"` uses a GPU if one is visible. Lower `max_epochs` for a quick smoke run.

In [11]:
model.train(
    max_epochs=200,
    batch_size=128,
    lr=1e-3,
    accelerator="auto",
    devices=1,
    patience=50,
    default_root_dir="runs/barrnaseq_gendiff",
)

/rds/user/wz369/hpc-work/LIBS/mamba/envs/GenDiff_env/lib/python3.9/site-packages/lightning_fabric/plugins/environments/slurm.py:165: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python3.9 /rds/user/wz369/hpc-work/LIBS/mamba/envs/GenDiff_ ...
  rank_zero_warn(
GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


IPU available: False, using: 0 IPUs


HPU available: False, using: 0 HPUs


You are using a CUDA device ('NVIDIA A100-SXM4-80GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


Missing logger folder: runs/barrnaseq_gendiff/lightning_logs


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



  | Name    | Type           | Params
-------------------------------------------
0 | model   | Epsilon_Linear | 2.9 M 
1 | loss_fn | SmoothL1Loss   | 0     
-------------------------------------------
2.9 M     Trainable params
0         Non-trainable params
2.9 M     Total params
11.576    Total estimated model params size (MB)


/rds/user/wz369/hpc-work/LIBS/mamba/envs/GenDiff_env/lib/python3.9/site-packages/pytorch_lightning/trainer/trainer.py:1609: PossibleUserWarning: The number of training batches (35) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.
  rank_zero_warn(


## 6. Save

Writes `model.pt` (state dict) and `attr.json` (setup, init params, token dict). Reload later with
`GenDiff.load(dir, adata)` — see tutorial 2.

In [12]:
model.save("runs/barrnaseq_gendiff/model")
print("saved to runs/barrnaseq_gendiff/model")

saved to runs/barrnaseq_gendiff/model


---
Next: **`2_downstream_analysis.ipynb`** — load this model and read off per-condition displacement fields,
top moving genes, trajectories, and a comparison to RNA velocity.